In [2]:
from pathlib import Path
import polars as pl

# === 0) Trỏ đúng thư mục chứa tất cả các .parquet ===
FOLDER = Path.cwd()

# === 1) Lấy danh sách file .parquet & tách theo tên ===
all_parquets = sorted(FOLDER.glob("*.parquet"))

# Quy tắc lọc tên: chỉnh lại nếu naming khác
user_files = [str(p) for p in all_parquets if "user" in p.name.lower()]
purchase_files = [str(p) for p in all_parquets if ("purchase" in p.name.lower()) or ("history" in p.name.lower())]
item_files = [str(p) for p in all_parquets if "item" in p.name.lower()]

# Kiểm tra nhanh kỳ vọng: user & purchase có >=1 file; item đúng 1 file
if not user_files:
    raise FileNotFoundError("Không tìm thấy file parquet nào thuộc nhóm 'user'. Kiểm tra lại tên file.")
if not purchase_files:
    raise FileNotFoundError("Không tìm thấy file parquet nào thuộc nhóm 'purchase_history'. Kiểm tra lại tên file.")
if len(item_files) != 1:
    raise ValueError(f"Nhóm 'item' phải có đúng 1 file parquet, hiện có {len(item_files)}: {item_files}")

# === 2) Load LAZY (không nuốt toàn bộ vào RAM) ===
users_lf = pl.scan_parquet(user_files)            # nhiều file user
purchases_lf = pl.scan_parquet(purchase_files)    # nhiều file purchase_history
items_lf = pl.scan_parquet(item_files[0])         # đúng 1 file item

# (Tùy chọn) In schema để xác nhận đã load đúng mà không collect toàn bộ
print("USER schema:", users_lf.schema)
print("PURCHASE schema:", purchases_lf.schema)
print("ITEM schema:", items_lf.schema)

# (Chỉ để kiểm tra nhanh 5 dòng đầu - có thể bỏ nếu bạn chỉ muốn 'load')
# print(users_lf.head(5).collect())
# print(purchases_lf.head(5).collect())
# print(items_lf.head(5).collect())

# Sau đây bạn đã có 3 LazyFrame:
#   users_lf, purchases_lf, items_lf
# Chỉ collect khi thật sự cần:
# df_users = users_lf.collect(streaming=True)
# df_purchases = purchases_lf.collect(streaming=True)
# df_items = items_lf.collect(streaming=True)


USER schema: Schema([('customer_id', Int32), ('gender', String), ('location', Int32), ('province', String), ('membership', String), ('timestamp', Int64), ('created_date', Datetime(time_unit='us', time_zone=None)), ('updated_date', Datetime(time_unit='us', time_zone=None)), ('sync_status_id', Int32), ('last_sync_date', Datetime(time_unit='us', time_zone=None)), ('sync_error_message', String), ('region', String), ('location_name', String), ('install_app', String), ('install_date', Int64), ('district', String), ('user_id', String), ('is_deleted', Boolean)])
PURCHASE schema: Schema([('timestamp', Int64), ('user_id', String), ('item_id', String), ('event_type', String), ('event_value', Decimal(precision=38, scale=4)), ('price', Decimal(precision=38, scale=4)), ('date_key', Int32), ('quantity', Int32), ('customer_id', Int32), ('created_date', Datetime(time_unit='us', time_zone=None)), ('updated_date', Datetime(time_unit='us', time_zone=None)), ('channel', String), ('payment', String), ('loca

C:\Users\ASUS\AppData\Local\Temp\ipykernel_15712\727344013.py:29: PerformanceWarning: Resolving the schema of a LazyFrame is a potentially expensive operation. Use `LazyFrame.collect_schema()` to get the schema without this warning.
  print("USER schema:", users_lf.schema)
C:\Users\ASUS\AppData\Local\Temp\ipykernel_15712\727344013.py:30: PerformanceWarning: Resolving the schema of a LazyFrame is a potentially expensive operation. Use `LazyFrame.collect_schema()` to get the schema without this warning.
  print("PURCHASE schema:", purchases_lf.schema)
C:\Users\ASUS\AppData\Local\Temp\ipykernel_15712\727344013.py:31: PerformanceWarning: Resolving the schema of a LazyFrame is a potentially expensive operation. Use `LazyFrame.collect_schema()` to get the schema without this warning.
  print("ITEM schema:", items_lf.schema)


In [4]:
print(users_lf.head(5).collect())

shape: (5, 18)
┌────────────┬────────┬──────────┬────────────┬───┬────────────┬──────────┬────────────┬───────────┐
│ customer_i ┆ gender ┆ location ┆ province   ┆ … ┆ install_da ┆ district ┆ user_id    ┆ is_delete │
│ d          ┆ ---    ┆ ---      ┆ ---        ┆   ┆ te         ┆ ---      ┆ ---        ┆ d         │
│ ---        ┆ str    ┆ i32      ┆ str        ┆   ┆ ---        ┆ str      ┆ str        ┆ ---       │
│ i32        ┆        ┆          ┆            ┆   ┆ i64        ┆          ┆            ┆ bool      │
╞════════════╪════════╪══════════╪════════════╪═══╪════════════╪══════════╪════════════╪═══════════╡
│ 14732      ┆ Nam    ┆ 155      ┆ Hồ Chí     ┆ … ┆ 1306281600 ┆ 7        ┆ e1e4820665 ┆ false     │
│            ┆        ┆          ┆ Minh       ┆   ┆            ┆          ┆ 2bf8c279ff ┆           │
│            ┆        ┆          ┆            ┆   ┆            ┆          ┆ 0206c69a80 ┆           │
│            ┆        ┆          ┆            ┆   ┆            ┆          ┆ 